In [ ]:
from ingest import fetch_handbook_documents

markdown_files = fetch_handbook_documents()
print(f"Fetched {len(markdown_files)} markdown files.")
if markdown_files:
    print("\nFirst file path:")
    print(markdown_files[0]["path"])

    print("\nFirst 500 characters of first file:")
    print(markdown_files[0]["content"][:500])
else:
    print("No markdown files found.")

In [ ]:
len(markdown_files)

In [ ]:
from ingest import save_documents_to_json
save_documents_to_json(markdown_files)

In [ ]:
import json
from pathlib import Path


def load_documents_from_json(
    input_path: str = "data/employee_handbook_documents.json",
) -> list[dict[str, str]]:
    path = Path(input_path)

    with path.open("r", encoding="utf-8") as f:
        documents = json.load(f)

    return documents

In [ ]:
documents = load_documents_from_json()
documents

In [ ]:
from tqdm.auto import tqdm
from embedder import Embedder

embed = Embedder()

vectors = embed.encode_batch([d["content"] for d in tqdm(documents)])

In [ ]:
from sqlitesearch import VectorSearchIndex

index = VectorSearchIndex(
    mode="ivf",
    keyword_fields=["content"],
    db_path="employee_handbook_vectors.db"
)

index.fit(vectors,documents)

In [ ]:
query = "Does the company provide insurance benefits to its employees?"
query_vector = embed.encode(query)

results = index.search(query_vector, num_results=5)

In [ ]:
results

In [ ]:
index.clear()
index.close()